In [ ]:
import sys
sys.path.append('../')

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import pandas as pd
import torch

from typing import Callable, Iterable, List, Tuple
from tokenizers import ByteLevelBPETokenizer

from src.config.paths import DATA_PATH
from src.config.lex import N_VOCAB, CHUNK_SIZE

# Dataset

In [9]:
df = pd.read_parquet(DATA_PATH.joinpath('dia-merged2.parquet'))

## Conversation Chunking

In [10]:
counts = df.groupby('conversation_id')['id'].count()
counts 

conversation_id
10           160
100          101
1004          36
1007         167
1010         131
            ... 
iq2_897     1087
iq2_9204    1182
iq2_9437    1184
iq2_9739    1290
iq2_9973    1379
Name: id, Length: 3151, dtype: int64

In [11]:
counts.max(), counts.mean(), counts.mode()[0], counts.min()

(np.int64(1665), np.float64(187.71405902887972), np.int64(92), np.int64(4))

In [12]:
for limit in [16, 32, 64]:
    outliers = (counts < limit)
    print(limit, '-', outliers.sum(), '-', f'{outliers.sum() / len(counts):%}')

16 - 45 - 1.428118%
32 - 198 - 6.283719%
64 - 807 - 25.610917%


I am torn between 32 and 64 right now but leaning 32
- Only 6.28% of convos would be dropped if i did chunking with a size of 32, 
- its reasonably large to provide good context to the average sentence within a chunk
- its not so big it would slow down training

Obviously samples from the non-outlier convos would be dropped too but I guess I can go back to 16 if its too bad

I guess if even after chunking I feel I have too much data, I might opt for 64, but that can definitely get rough on training so probably gonna stick with 32

In [13]:
res = []

chunk_idx = -1
in_chunk_idx = 0
last_convo = None

for _, row in df.iterrows():
    if not last_convo or last_convo != row.conversation_id:
        last_convo = row.conversation_id
        chunk_idx += 1
        in_chunk_idx = 0

    res.append(chunk_idx)

    in_chunk_idx += 1 
    if CHUNK_SIZE == in_chunk_idx:
        in_chunk_idx = 0
        chunk_idx += 1
        
df['chunk_id'] = res

Removing Semi-Filled Chunks

In [14]:
counts = df.groupby('chunk_id')['id'].count()
filled_chunks = counts[counts == CHUNK_SIZE].index.to_numpy()

df = df[df['chunk_id'].isin(filled_chunks)]

In [15]:
ids_oton = {v:k for k,v in enumerate(df['chunk_id'].unique())}
df['chunk_id'] = df['chunk_id'].apply(lambda x: ids_oton[x])

Removing Single-Speaker Chunks

In [16]:
n_chunks = len(df['chunk_id'].unique())
single_speaker_chunks = []

for i in range(n_chunks):
    chunk = df[df['chunk_id'] == i]
    if len(chunk['speaker'].unique()) == 1:
        single_speaker_chunks.append(i)

df = df[~df['chunk_id'].isin(single_speaker_chunks)]

In [17]:
ids_oton = {v:k for k,v in enumerate(df['chunk_id'].unique())}
df['chunk_id'] = df['chunk_id'].apply(lambda x: ids_oton[x])

In [18]:
df.groupby('root')['id'].count()

root
iq2    107936
sup    203488
ten    207584
Name: id, dtype: int64

## Triplet Mining

In [19]:
df

,id,conversation_id,speaker,text,root,chunk_id
0,1681_0.q_0,1681,REPORTER,I think this is your biggest success right now...,ten,0
1,1681_0.a_0,1681,Kei Nishikori,Yeah.,ten,0
2,1681_1.q_0,1681,REPORTER,How would you describe it?,ten,0
2,1681_1.q_1,1681,REPORTER,Is it fantastic for you?,ten,0
3,1681_1.a_0,1681,Kei Nishikori,"Yeah, I'm pretty happy, but it was --",ten,0
...,...,...,...,...,...,...
505512,24959__4_000_21,24959,peter_k_stris,"Ah, the participants, the beneficiaries, they'...",sup,16218
505513,24959__4_000_22,24959,peter_k_stris,The plan is the beneficiary.,sup,16218
505514,24959__4_000_23,24959,peter_k_stris,"If he's right, we lose.",sup,16218
505515,24959__4_000_24,24959,peter_k_stris,"But he's obviously wrong, because the benefici...",sup,16218


In [20]:
n_chunks = len(df['chunk_id'].unique())
chunks = []

for i in range(n_chunks):
    print(f'\r[{i+1}/{n_chunks}]', end='')
    chunk = df[df['chunk_id'] == i].set_index('id')

    neg_proposal = chunk.index.to_numpy()
    pos_proposal = neg_proposal.copy()

    neg_result = np.zeros_like(neg_proposal)
    pos_result = neg_result.copy()

    while not np.all(pos_result) or not np.all(neg_result):
        neg_proposal = np.roll(neg_proposal, 1)
        pos_proposal = np.roll(pos_proposal, -1)

        pos_correct = chunk['speaker'] == chunk.loc[pos_proposal, 'speaker'].to_numpy()
        neg_correct = chunk['speaker'] != chunk.loc[neg_proposal, 'speaker'].to_numpy()

        pos_result[pos_correct] = pos_proposal[pos_correct]
        neg_result[neg_correct] = neg_proposal[neg_correct]
    
    chunk['pos_id'] = pos_result
    chunk['neg_id'] = neg_result

    chunks.append(chunk)
    
df = pd.concat(chunks)

[16219/16219]

In [30]:
df = df[['chunk_id', 'text', 'pos_id', 'neg_id']]
df.to_parquet(DATA_PATH.joinpath('dia-final.parquet'))